# 16S Analyses of the Longitudinal Acne Study
## Alpha Diversity Plots

Date last updated: 1/7/25

Notebook authors: Yang Chen, Britta De Pessemier, and Tyler Myers

This notebook plots the following:

- 16S V1-V3 and V4 Shannon alpha diversity plots at ASV level from rarefied abundance tables
- 16S V1-V3 and V4 Faith PD alpha diversity plots at ASV level from rarefied abundance tables

In [1]:
# Import Python packages
import re
import pandas as pd
import numpy as np
import biom
from biom import Table
from biom.util import biom_open
from biom import load_table
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import cycle
import os
from matplotlib.colors import ListedColormap
from matplotlib.colors import to_rgba
from skbio.diversity import alpha_diversity
from skbio.diversity.alpha import shannon
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
from skbio import TreeNode
from skbio.diversity.alpha import faith_pd

In [2]:
# Load the metadata
metadata_path = '../Metadata/metadata_final_22102024.tsv'
metadata = pd.read_csv(metadata_path, sep='\t')
metadata

,SampleID,c_zone,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_face,zone,sample_type,planned_study_day_of_visit,visual_assessment_in_vivo_number_of_inflammatory_lesions_face,day,subject_randomization_number,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_cheek_right,...,a,cohort,subject_randomization_id,class,subject_ID,subject_ID_CC,zone_CC,group,severity_level,severity_group
0,LAMI.RD308.D16.C1,C1,not applicable,Lesional,skin,Day 16,not applicable,16,308,not applicable,...,33.765638,acne,RD308,acne,PP_308,PP_308C1,Lesional_C1,Acne_L,moderate,moderate Acne_L
1,LAMI.RD310.D21.C1,C1,72,Lesional,skin,Day 21,36,21,310,17,...,31.919478,acne,RD310,acne,PP_310,PP_310C1,Lesional_C1,Acne_L,moderate,moderate Acne_L
2,LAMI.RD305.D21.C3,C3,69,Non-lesional,skin,Day 21,26,21,305,25,...,22.152503,acne,RD305,healthy,PP_305,PP_305C3,Non-lesional_C3,Acne_NL,absent,absent Acne_NL
3,LAMI.RD306.D18.C2,C2,not applicable,Lesional,skin,Day 18,not applicable,18,306,not applicable,...,27.397918,acne,RD306,acne,PP_306,PP_306C2,Lesional_C2,Acne_L,low,low Acne_L
4,LAMI.RD306.D7.C2,C2,90,Lesional,skin,Day 7,13,7,306,23,...,28.819451,acne,RD306,acne,PP_306,PP_306C2,Lesional_C2,Acne_L,moderate,moderate Acne_L
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,LAMI.RD317.D21.C1,C1,77,Lesional,skin,Day 21,19,21,317,20,...,21.946648,acne,RD317,acne,PP_317,PP_317C1,Lesional_C1,Acne_L,low,low Acne_L
262,LAMI.RD001.D0.C1,C1,not applicable,Non-lesional,skin,Day 0,not applicable,0,1,not applicable,...,26.344240,control,RD001,healthy,PP_1,PP_1C1,Non-lesional_C1,Healthy,absent,absent Healthy
263,LAMI.RD014.D14.C2,C2,not applicable,Non-lesional,skin,Day 14,not applicable,14,14,not applicable,...,16.359699,control,RD014,healthy,PP_14,PP_14C2,Non-lesional_C2,Healthy,absent,absent Healthy
264,LAMI.RD314.D0.C1,C1,55,Lesional,skin,Day 0,31,0,314,16,...,22.494605,acne,RD314,acne,PP_314,PP_314C1,Lesional_C1,Acne_L,low,low Acne_L


In [3]:
# Define paths to tables
biom_paths = {
    'V1-V3': '../Data/16S/Tables/from_Qiita/179426_feature-table_16S_V1V3_rare-11054_sampleIDfixed.biom',
    'V4': '../Data/16S/Tables/from_Qiita/174951_feature-table_16S_V4_rare-3769_sampleIDfixed.biom'
}

In [4]:
# Function to load BIOM table, collapse by taxa, sort rows by row sum
def load_biom_table(biom_path, metadata_path):
    # Load metadata as a DataFrame from the file path
    metadata = pd.read_csv(metadata_path, sep='\t')

    # Load BIOM table and convert to a DataFrame
    table = biom.load_table(biom_path)
    df = pd.DataFrame(table.matrix_data.toarray(),
                      index=table.ids(axis='observation'),
                      columns=table.ids(axis='sample'))
    
    # Sort rows by row sum in descending order
    df['row_sum'] = df.sum(axis=1)
    df = df.sort_values(by='row_sum', ascending=False)
    
    # Drop the 'row_sum' column before proceeding
    df = df.drop(columns=['row_sum'])
    
    return df

In [5]:
# Use sample samples from V1-V3 and V4 for alpha diversity analyses
def get_clean_sample_ids(biom_path):
    table = load_table(biom_path)
    sample_ids = pd.Series(list(table.ids(axis='sample')))

    sample_ids = (
        sample_ids
        .astype(str)
        .str.replace(r'^.*?(?=LAMI)', '', regex=True)
    )

    return set(sample_ids)

In [6]:
# Subset tables for paired-sample consistency
biom_v1v3 = biom_paths['V1-V3']
biom_v4   = biom_paths['V4']

samples_v1v3 = get_clean_sample_ids(biom_v1v3)
samples_v4   = get_clean_sample_ids(biom_v4)

shared_samples = samples_v1v3.intersection(samples_v4)

print(f"Shared samples between V1–V3 and V4: {len(shared_samples)}")

Shared samples between V1–V3 and V4: 190


In [7]:
# Load metadata
metadata = pd.read_csv(metadata_path, sep='\t')

# Ensure SampleID is string
metadata['SampleID'] = metadata['SampleID'].astype(str)

# Keep only samples shared between V1–V3 and V4
md_shared = metadata[metadata['SampleID'].isin(shared_samples)]

# Count per group
group_counts = (
    md_shared
    .groupby('group')
    .size()
    .reset_index(name='n')
)

print(group_counts)

     group    n
0   Acne_L  123
1  Acne_NL   42
2  Healthy   19


## Shannon Alpha Diversity

In [8]:
def calculate_shannon_alpha_diversity_and_plot(
    outdir,
    biom_path,
    metadata_path,
    group_col='group',
    title_suffix='',
    keep_samples=None,
    save_biom_path=None
):
    """
    Compute Shannon alpha diversity from a BIOM table and plot by group.
    Optionally save the filtered BIOM table actually used for the analysis.
    """

    # Load BIOM table and metadata
    table = load_table(biom_path)
    metadata = pd.read_csv(metadata_path, sep='\t')

    # Ensure paired-sample consistency
    if keep_samples is not None:
        keep_samples = set(keep_samples)
        samples_in_table = {
            sid for sid in table.ids(axis='sample')
            if re.sub(r'^.*?(?=LAMI)', '', str(sid)) in keep_samples
        }
        table = table.filter(samples_in_table, axis='sample', inplace=False)

    # Save the filtered BIOM used for alpha diversity (optional)
    if save_biom_path is not None:
        os.makedirs(os.path.dirname(save_biom_path), exist_ok=True)
        with h5py.File(save_biom_path, "w") as f:
            table.to_hdf5(f, "alpha-diversity-input")

    # Compute Shannon diversity
    shannon_df = pd.DataFrame({
        'sample_id': list(table.ids(axis='sample')),
        'Shannon': [
            shannon(table.data(sid, axis='sample'))
            for sid in table.ids(axis='sample')
        ]
    })

    # Clean BIOM sample IDs (remove everything before 'LAMI')
    shannon_df['sample_id'] = (
        shannon_df['sample_id']
        .astype(str)
        .str.replace(r'^.*?(?=LAMI)', '', regex=True)
    )

    # Prepare metadata IDs
    if 'SampleID' not in metadata.columns:
        raise ValueError("Metadata must contain a 'SampleID' column")

    metadata['SampleID'] = metadata['SampleID'].astype(str)

    # Merge BIOM and metadata
    metadata = metadata.merge(
        shannon_df,
        left_on='SampleID',
        right_on='sample_id',
        how='inner'
    )

    if metadata.empty:
        print(f"[WARN] No matching samples after merge for {title_suffix}")
        return

    # Clean columns
    metadata = metadata.dropna(subset=[group_col, 'Shannon'])
    metadata[group_col] = metadata[group_col].astype(str)

    # Color palettes
    palette = {
        'Healthy': '#3333B3',
        'Acne_NL': '#5cbccb',
        'Acne_L': '#f16c52'
    }

    darker_palette = {
        'Healthy': '#23238f',
        'Acne_NL': '#44a7b6',
        'Acne_L': '#d8543e'
    }

    severity_palette = {
        'low': '#f3b7a6',
        'moderate': '#e07b63',
        'high': '#b7372a'
    }

    # Group order
    desired_order = ['Healthy', 'Acne_NL', 'Acne_L']
    present_groups = metadata[group_col].unique().tolist()
    plot_order = [g for g in desired_order if g in present_groups]

    if not plot_order:
        print(f"[WARN] No groups to plot for {title_suffix}")
        return

    # Create figure
    fig, ax = plt.subplots(figsize=(4, 6))

    # Boxplot
    sns.boxplot(
        data=metadata,
        x=group_col,
        y='Shannon',
        order=plot_order,
        palette=palette,
        showcaps=True,
        boxprops=dict(edgecolor='black', linewidth=1.2),
        whiskerprops=dict(color='black', linewidth=1.2),
        medianprops=dict(color='black', linewidth=1.5),
        showfliers=False,
        ax=ax
    )

    # Stripplot for Healthy + Acne_NL
    non_lesional = metadata[metadata[group_col] != 'Acne_L']
    sns.stripplot(
        data=non_lesional,
        x=group_col,
        y='Shannon',
        order=plot_order,
        jitter=True,
        palette=darker_palette,
        size=4,
        alpha=0.9,
        ax=ax,
        edgecolor='black',
        linewidth=0.4,
        zorder=2
    )

    # Stripplot for Acne_L by severity
    if 'severity_level' in metadata.columns:
        acne_l = metadata[metadata[group_col] == 'Acne_L']
        if not acne_l.empty:
            sns.stripplot(
                data=acne_l,
                x=group_col,
                y='Shannon',
                order=plot_order,
                hue='severity_level',
                hue_order=['low', 'moderate', 'high'],
                palette=severity_palette,
                jitter=True,
                dodge=False,
                size=4,
                alpha=0.9,
                ax=ax,
                edgecolor='black',
                linewidth=0.4,
                legend=True,
                zorder=3
            )

            handles, labels = ax.get_legend_handles_labels()

            label_map = {
                'low': 'Low (1–2)',
                'moderate': 'Mod (3–4)',
                'high': 'High (5–6)'
            }

            new_labels = [label_map[l] for l in labels]

            for h in handles:
                h.set_sizes([16])
                h.set_edgecolor('black')
                h.set_linewidth(0.6)

            ax.legend(
                handles,
                new_labels,
                title="Severity",
                fontsize=8,
                title_fontsize=9,
                frameon=True
            )

    # Helper to annotate p-values
    def add_pvalue(ax, x1, x2, y, p):
        if p < 1e-3:
            stars = "***"
        elif p < 1e-2:
            stars = "**"
        elif p < 5e-2:
            stars = "*"
        else:
            stars = ""

        if p < 1e-3:
            p_txt = f"{p:.2e}"
        else:
            p_txt = f"{p:.2g}"

        label = f"{stars}  p={p_txt}".strip()

        ax.plot([x1, x1, x2, x2], [y, y + 0.05, y + 0.05, y],
                lw=1.2, c='black')

        ax.text((x1 + x2) / 2, y + 0.06, label,
                ha='center', va='bottom', fontsize=8)

    # Statistical testing (FDR-corrected)
    ymax = metadata['Shannon'].max()
    step = ymax * 0.1
    current_y = ymax + step

    comparisons = [
        ('Healthy', 'Acne_NL'),
        ('Healthy', 'Acne_L'),
        ('Acne_NL', 'Acne_L')
    ]

    raw_pvals = []
    valid_pairs = []

    for g1, g2 in comparisons:
        if g1 in plot_order and g2 in plot_order:
            v1 = metadata.loc[metadata[group_col] == g1, 'Shannon']
            v2 = metadata.loc[metadata[group_col] == g2, 'Shannon']
            if len(v1) > 0 and len(v2) > 0:
                p = mannwhitneyu(v1, v2, alternative='two-sided').pvalue
                raw_pvals.append(p)
                valid_pairs.append((g1, g2))

    if raw_pvals:
        _, pvals_adj, _, _ = multipletests(raw_pvals, method='fdr_bh')
        for (g1, g2), p_adj in zip(valid_pairs, pvals_adj):
            x1 = plot_order.index(g1)
            x2 = plot_order.index(g2)
            add_pvalue(ax, x1, x2, current_y, p_adj)
            current_y += step * 0.8

    # X-axis labels with sample counts
    group_counts = metadata.groupby(group_col).size().to_dict()

    label_map = {
        'Healthy': 'Healthy\nSkin',
        'Acne_NL': 'Acne\nNon-lesional',
        'Acne_L': 'Acne\nLesional'
    }

    xtick_labels = [
        f"{label_map[g]}\n(n={group_counts.get(g, 0)})"
        for g in plot_order
    ]

    ax.set_xticklabels(xtick_labels)

    # Final styling
    ax.set_title(f"16S rRNA ({title_suffix}) Alpha Diversity")
    ax.set_xlabel("")
    ax.set_ylabel("Shannon Index")

    sns.despine()
    plt.tight_layout()

    # Save figure
    os.makedirs(outdir, exist_ok=True)
    fname = f"../Figures/Supplementary/Suppl_Figure_3/Shannon_alpha_diversity_{title_suffix.replace(' ', '_')}.png"
    fpath = os.path.join(outdir, fname)

    plt.savefig(fpath, dpi=600, bbox_inches='tight')
    plt.close(fig)


In [9]:
# Plot Shannon Diversity plots for both V1-V3 and V4
for key, biom_path in biom_paths.items():
    calculate_shannon_alpha_diversity_and_plot(
        outdir='../Figures/',
        biom_path=biom_path,
        metadata_path=metadata_path,
        group_col='group',
        title_suffix=key,
        keep_samples=shared_samples
    )

/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_22659/3364074561.py:120: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(
/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na', True):
/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na', True):
/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/_oldcore.py:1075: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple

## Faith Phylogenetic Alpha Diversity

In [10]:
def compute_faith_pd(biom_path, tree_path):
    table = load_table(biom_path)

    # Load tree
    tree = TreeNode.read(tree_path)

    # Convert BIOM to dense dataframe
    counts = table.to_dataframe(dense=True).T

    # Faith PD per sample
    faith = {
        sample_id: faith_pd(
            counts.loc[sample_id].values,
            counts.columns,
            tree
        )
        for sample_id in counts.index
    }

    df = (
        pd.DataFrame.from_dict(faith, orient='index', columns=['Faith_PD'])
        .reset_index()
        .rename(columns={'index': 'sample_id'})
    )

    # Clean sample IDs (same logic as before)
    df['sample_id'] = (
        df['sample_id']
        .astype(str)
        .str.replace(r'^.*?(?=LAMI)', '', regex=True)
    )

    return df


In [11]:
faith_v1v3 = compute_faith_pd(
    biom_paths['V1-V3'],
    tree_path='../Data/16S/Trees/16S_V1-V3_173980_insertion_tree.relabelled.nwk'
)

faith_v4 = compute_faith_pd(
    biom_paths['V4'],
    tree_path='../Data/16S/Trees/16S_V4_173620_insertion_tree.relabelled.nwk'
)


In [12]:
faith_v1v3 = faith_v1v3.set_index('sample_id')
faith_v1v3.index.name = None
faith_v1v3

,Faith_PD
LAMI.RD001.D0.C1,6.256115
LAMI.RD001.D0.C2,7.205853
LAMI.RD001.D14.C1,7.059174
LAMI.RD001.D28.C1,5.887058
LAMI.RD002.D14.C1,5.599563
...,...
LAMI.RD318.D28.C3,2.005455
LAMI.RD319.D14.C1,2.830915
LAMI.RD319.D14.C2,2.299421
LAMI.RD319.D9.C1,2.507325


In [13]:
faith_v4 = faith_v4.set_index('sample_id')
faith_v4.index.name = None
faith_v4

,Faith_PD
LAMI.RD001.D0.C1,10.279681
LAMI.RD001.D14.C1,11.114928
LAMI.RD004.D0.C2,8.344655
LAMI.RD001.D0.C2,11.709739
LAMI.RD004.D28.C1,10.851166
...,...
LAMI.RD319.D16.C2,4.465293
LAMI.RD319.D28.C1,4.953941
LAMI.RD318.D9.C2,5.618270
LAMI.RD319.D28.C2,3.925977


In [14]:
# Subset to same shared sample ids
common_idx = faith_v1v3.index.intersection(faith_v4.index)

faith_v1v3_shared = faith_v1v3.loc[common_idx].copy()
faith_v4_shared = faith_v4.loc[common_idx].copy()

In [15]:
def plot_faithPD_alpha_diversity(
    df,                 # faith_v1v3_shared OR faith_v4_shared
    metadata_path,
    group_col,
    title_suffix,
    outdir,
    key                 # 'V1-V3' or 'V4'
):
    import os
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    from scipy.stats import mannwhitneyu
    from statsmodels.stats.multitest import multipletests

    # Load and clean metadata
    metadata = pd.read_csv(metadata_path, sep='\t')
    metadata['SampleID'] = (
        metadata['SampleID']
        .astype(str)
        .str.replace(r'^.*?(?=LAMI)', '', regex=True)
    )
    metadata = metadata.set_index('SampleID')

    # Align metadata and Faith PD
    common_idx = metadata.index.intersection(df.index)
    metadata = metadata.loc[common_idx].copy()
    df = df.loc[common_idx].copy()

    # If df is a Series, convert to DataFrame
    if isinstance(df, pd.Series):
        df = df.to_frame(name='Faith_PD')

    # Merge Faith_PD explicitly
    metadata['Faith_PD'] = df.iloc[:, 0]

    # Severity categories
    metadata['severity_category'] = pd.cut(
        metadata['local_lesion_severity'],
        bins=[0, 2, 4, 6],
        labels=['Low', 'Mod', 'High'],
        include_lowest=True
    )

    plot_order = ['Healthy', 'Acne_NL', 'Acne_L']
    plot_order = [g for g in plot_order if g in metadata[group_col].unique()]

    palette = {
        'Healthy': '#23238f',
        'Acne_NL': '#44a7b6',
        'Acne_L': '#d8543e'
    }

    severity_palette = {
        'Low': '#F1948A',
        'Mod': '#EC7063',
        'High': '#C0392B'
    }

    # Plot
    fig, ax = plt.subplots(figsize=(3.5, 6))

    sns.boxplot(
        data=metadata,
        x=group_col,
        y='Faith_PD',
        order=plot_order,
        palette=palette,
        ax=ax,
        fliersize=0,
        linewidth=1.2
    )

    sns.stripplot(
        data=metadata,
        x=group_col,
        y='Faith_PD',
        order=plot_order,
        palette=palette,
        ax=ax,
        jitter=True,
        dodge=False,
        size=5,
        edgecolor='black',
        linewidth=0.6
    )

    # Overlay severity for Acne_L
    acne_l = metadata[metadata[group_col] == 'Acne_L']
    if not acne_l.empty:
        sns.stripplot(
            data=acne_l,
            x=group_col,
            y='Faith_PD',
            hue='severity_category',
            hue_order=['Low', 'Mod', 'High'],
            palette=severity_palette,
            ax=ax,
            jitter=True,
            dodge=False,
            size=5,
            edgecolor='black',
            linewidth=0.6
        )

        handles, labels = ax.get_legend_handles_labels()
        handles = handles[-3:]
        labels = ['Low (1–2)', 'Mod (3–4)', 'High (5–6)']

        for h in handles:
            h.set_sizes([16])
            h.set_edgecolor('black')
            h.set_linewidth(0.6)

        legend_pos = {
            'V1-V3': dict(loc='upper right', bbox_to_anchor=(0.84, 0.9)),
            'V4':    dict(loc='upper left',  bbox_to_anchor=(0.05, 0.19)),
        }
        pos = legend_pos.get(key, dict(loc='upper right', bbox_to_anchor=(0.98, 0.98)))

        ax.legend(
            handles,
            labels,
            title='Severity',
            fontsize=8,
            title_fontsize=9,
            frameon=True,
            borderaxespad=0.3,
            **pos
        )
    else:
        if ax.legend_:
            ax.legend_.remove()

    # X-axis labels with counts
    label_map = {
        'Healthy': 'H',
        'Acne_NL': 'ANL',
        'Acne_L': 'AL'
    }

    group_counts = metadata.groupby(group_col).size().to_dict()
    xticklabels = [
        f"{label_map[g]}\n(n={group_counts.get(g, 0)})"
        for g in plot_order
    ]

    ax.set_xticklabels(xticklabels, fontsize=11)
    ax.set_xlabel('')
    ax.set_ylabel('Faith PD')
    ax.set_title(f'16S rRNA ({title_suffix}) Alpha Diversity')

    # Statistics (Mann–Whitney + FDR)
    comparisons = []
    pvals = []

    for i in range(len(plot_order)):
        for j in range(i + 1, len(plot_order)):
            g1, g2 = plot_order[i], plot_order[j]
            v1 = metadata.loc[metadata[group_col] == g1, 'Faith_PD']
            v2 = metadata.loc[metadata[group_col] == g2, 'Faith_PD']
            _, p = mannwhitneyu(v1, v2, alternative='two-sided')
            comparisons.append((i, j))
            pvals.append(p)

    reject, pvals_corr, _, _ = multipletests(pvals, method='fdr_bh')

    y_base = metadata['Faith_PD'].max()
    step = (metadata['Faith_PD'].max() - metadata['Faith_PD'].min()) * 0.05

    for (i, j), p_corr in zip(comparisons, pvals_corr):

        stars = (
            '***' if p_corr < 0.001 else
            '**'  if p_corr < 0.01  else
            '*'   if p_corr < 0.05 else
            ''
        )

        p_str = f"{p_corr:.3f}"
        y = y_base + step

        ax.plot(
            [i, i, j, j],
            [y, y + step * 0.1, y + step * 0.1, y],
            lw=1,
            c='black'
        )

        label = f"{stars} p={p_str}" if stars else f"p={p_str}"
        ax.text(
            (i + j) / 2,
            y + step * 0.2,
            label,
            ha='center',
            va='bottom',
            fontsize=9
        )

        y_base += step * 1.2

    sns.despine()
    plt.tight_layout()

    os.makedirs(outdir, exist_ok=True)
    plt.savefig(
        os.path.join(outdir, f'FaithPD_alpha_diversity_{key}.png'),
        dpi=600,
        bbox_inches='tight'
    )

    plt.close(fig)


In [16]:
# Plot Faith PD for V1–V3 (shared samples only)
plot_faithPD_alpha_diversity(
    df=faith_v1v3_shared,
    metadata_path=metadata_path,
    group_col='group',
    title_suffix='V1–V3',
    outdir='../Figures/Main/Figure_3/',
    key='V1-V3'
)

# Plot Faith PD for V4 (shared samples only)
plot_faithPD_alpha_diversity(
    df=faith_v4_shared,
    metadata_path=metadata_path,
    group_col='group',
    title_suffix='V4',
    outdir='../Figures/Main/Figure_3/',
    key='V4'
)


/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_22659/1490201671.py:74: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(
/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na', True):
/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na', True):
/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/_oldcore.py:1075: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple 